# GreenTest: Start Here with Bash & Python

This is the **python** GreenTest from [Ecology Computing](https://ecology-computing.com). Every language in this repo follows the
same pattern: bootstrap the language, generate a small static site, serve
it locally,  and verify it's being served correctly. From there most basic python scripts and apps should work as expected. This repo also includes some other basic linux apps and config we'll need.

Each greenTest notebook verifies [vanilla-compost](https://github.com/EcologyComputing/vanilla-compost),
the simplest possible static website. All it does is read a a directory of posts written as markdown files, generate a posts.html page to link to them, then display the markdown in the browser.

If this is your first time in a terminal, writing bash or python, using git,
looking at raw HTML, or you're just looking for a quick way to smoketest a bash based python environment, then this repo is for you!

Run the cells in order, top to bottom, using Shift+Enter. `generate_posts.py`
has no dependencies beyond the python standard library, so there's nothing
to `pip install` for the test itself. (If you can't even open this notebook
yet, run `bootstrap.sh` in this same directory first.)

## 0. Low-level bootstrapping & Dev environment setup

Before we check the repositories, let's verify your system has basic command-line tools configured. You should have run bootstrap.sh from the commandline to start this notebook, which would have installed or updated.
- [cURL](https://curl.se/docs/tutorial.html) is more or less for downloading and uploading on the command line
- [python3](https://en.wikipedia.org/wiki/Python_(programming_language)) The wikipedia page about the language, in case your _really_ new to python development.
- [python3-pip] is python's dependency manager. We needed pip to [install jupyter](https://jupyter.org/install), the package running this notebook.
- [python3-venv] creates local virtual environments for python projects so the system level python install isn't disturbed. In other words, jupyter is available in your venv, but not from your OS level python installation. 

### Git Identity Check
Git needs to know who you are to record commits. Let's check if `user.name` and `user.email` are set globally.

In [1]:
%%bash
# Check git config
NAME=$(git config --global user.name || echo "")
EMAIL=$(git config --global user.email || echo "")

if [ -z "$NAME" ] || [ -z "$EMAIL" ]; then
    echo "Git identity is not configured globally."
    echo "setting to name name and email."
    NAME="ecology-chris"
    EMAIL="chris@ecology-computing.com"
    git config --global user.name "$NAME"
    git config --global user.email "$EMAIL"
    git config --global init.defaultBranch main
fi
    echo "Git is configured:"
    echo "  Name:  $NAME"
    echo "  Email: $EMAIL"


Git is configured:
  Name:  ecology-chris
  Email: chris@ecology-computing.com


### GitHub Authentication Check
To pull and push code, you need SSH keys or the GitHub CLI (`gh`). 
* **SSH Keys** (Universal & Minimalist): Recommended by Ecology Computing conventions. Works out-of-the-box on all git servers without installing extra tools.
* **GitHub CLI (`gh`)** (Interactive & Convenient): Great if you are only using GitHub and want automated SSH/key setup via a browser login flow.

In [2]:
%%bash
# Test SSH connection to GitHub
echo "Testing SSH connection to GitHub..."
ssh -T git@github.com 2>&1 | head -n 1 || true

# Check if gh CLI is installed
if which gh >/dev/null 2>&1; then
    echo "Found gh CLI: $(gh --version | head -n 1)"
    gh auth status || true
else
    echo "gh CLI is not installed (optional)."
fi

Testing SSH connection to GitHub...
ssh_askpass: exec(/usr/bin/ssh-askpass): No such file or directory
Found gh CLI: gh version 2.96.0 (2026-07-02)
github.com
  ✓ Logged in to github.com account ecology-chris (/home/eco/.config/gh/hosts.yml)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo', 'workflow'


### Setting up GitHub Authentication
* To set up **SSH Keys**, run the cell below to generate a key pair and print your public key.
* To set up **GitHub CLI**, run `gh auth login` in your terminal.

In [3]:
%%bash
# Generate SSH Key (Uncomment and run if needed - replace email with your email)
# ssh-keygen -t ed25519 -C "your.email@example.com" -N "" -f ~/.ssh/id_ed25519

# If key exists, print it so you can copy/paste it to https://github.com/settings/keys
if [ -f ~/.ssh/id_ed25519.pub ]; then
    echo "Your SSH Public Key (copy the entire line below and add to GitHub settings):"
    cat ~/.ssh/id_ed25519.pub
else
    echo "No default ~/.ssh/id_ed25519.pub key found. Uncomment the command above to generate one."
fi

No default ~/.ssh/id_ed25519.pub key found. Uncomment the command above to generate one.


In [ ]:
%%bash 
# install github cli

(type -p curl >/dev/null || (sudo apt update && sudo apt install curl -y)) \
	&& sudo mkdir -p -m 755 /etc/apt/keyrings \
	&& out=$(mktemp) && curl -fsSL -o $out https://cli.github.com/packages/githubcli-archive-keyring.gpg \
	&& cat $out | sudo tee /etc/apt/keyrings/githubcli-archive-keyring.gpg > /dev/null \
	&& sudo chmod go+r /etc/apt/keyrings/githubcli-archive-keyring.gpg \
	&& sudo mkdir -p -m 755 /etc/apt/sources.list.d \
	&& echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | sudo tee /etc/apt/sources.list.d/github-cli.list > /dev/null \
	&& sudo apt update \
	&& sudo apt install gh -y

## 1. Point this notebook at your vanilla-compost clone

GreenTest assumes you cloned vanilla-compost as a sibling of this repo, so
the same folder that has `greenTest/` should also have `vanilla-compost/`.
This cell and the one below runs in **bash**, the language of the terminal, not python.
(In a notebook, a cell starting with `%%bash` switches to bash for that one cell.) 
If you're clone of vanilla compost lives somewhere else, change the path in the bash script below so it gets saved correctly as an environment variable.

## 2. Confirm the repo is cloned

This next script is also in bash. It checks that a couple of expected files exist, then checks that
the folder is a real `git clone`, or a full copy of the project's history, and
not just a folder of files someone emailed you. 

In [4]:
%%bash
export VANILLA_COMPOST="../../vanilla-compost"
echo "Testing vanilla-compost at: $VANILLA_COMPOST"

# A "git clone" is just a folder with a hidden .git directory tracking history.
if [ -f "$VANILLA_COMPOST/README.md" ] && [ -f "$VANILLA_COMPOST/src/generate_posts.py" ]; then
    echo "Repo layout looks right (found README.md and src/generate_posts.py)."
else
    echo "That path doesn't look like a vanilla-compost clone: $VANILLA_COMPOST"
    echo "Clone it first: git clone https://github.com/EcologyComputing/vanilla-compost.git"
    exit 1
fi

if git -C "$VANILLA_COMPOST" rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    echo "Confirmed: this is a real git clone, not just a folder of files."
    git -C "$VANILLA_COMPOST" remote get-url origin 2>/dev/null && echo "Its 'origin' remote points there ^ - that's where 'git pull' and 'git push' talk to." || echo "No 'origin' remote set - fine if you haven't pushed anywhere yet."
else
    echo "Warning: no .git found there. Fine for a quick trial, but you won't be able to track changes or send posts back via pull request without a real clone."
fi

Testing vanilla-compost at: ../../vanilla-compost
Repo layout looks right (found README.md and src/generate_posts.py).
Confirmed: this is a real git clone, not just a folder of files.
https://github.com/EcologyComputing/vanilla-compost.git
Its 'origin' remote points there ^ - that's where 'git pull' and 'git push' talk to.


## 3. Leave a note for this run

Edit the text `"""` marks below with anything about this run.
This is your own "Hello, World!". That's just a Python string
(text between quote marks). Editing it doesn't require knowing any python
beyond "this is text I can change." It gets appended to `greenTest-Message.md`
(with a timestamp) so there's a running record, not just a pass/fail.

In [5]:
notes = """this is working better!!!!"""

In [6]:
import os
import shutil
from datetime import datetime

log_path = "greenTest-Message.md"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"## {timestamp}\n\n{notes.strip()}\n\n")

print(f"Notes appended to {log_path}")

# Copy greenTest-Message.md to posts/ directory in vanilla-compost
VANILLA_COMPOST = os.environ.get("VANILLA_COMPOST", "../../vanilla-compost")
dest_dir = os.path.join(VANILLA_COMPOST, "src", "posts")
os.makedirs(dest_dir, exist_ok=True)
shutil.copy(log_path, os.path.join(dest_dir, log_path))
print(f"Copied {log_path} to {dest_dir}/")


Notes appended to greenTest-Message.md
Copied greenTest-Message.md to ../../vanilla-compost/src/posts/


## 4. Generate `posts.html`

This cell runs vanilla-compost's `generate_posts.py` - a small **python**
program (the same language powering this notebook) that reads every file
in its `src/posts/` and writes `src/posts.html` listing them. New to
Python? The [official tutorial](https://docs.python.org/3/tutorial/) is a
solid, free starting point.

In [7]:
import subprocess

result = subprocess.run(
    ["python3", os.path.join(VANILLA_COMPOST, "src", "generate_posts.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("generate_posts.py failed - see output above.")

Generated posts.html with 2 posts using html_template.html



Peek at what it generated - this is raw **HTML**, the markup language every
web page is built from. Curious what the tags mean?
[MDN's HTML docs](https://developer.mozilla.org/en-US/docs/Web/HTML) are the
standard reference.

In [8]:
with open(os.path.join(VANILLA_COMPOST, "src", "posts.html"), encoding="utf-8") as f:
    generated = f.read()

# show just the generated post list, not the whole page
start = generated.find('<p class="lead">')
end = generated.find('</p>', start) + len('</p>')
print(generated[start:end] if start != -1 else generated)

<p class="lead">
  <ul>
    <li><a href="post.html?post=greenTest-Message">Greentest Message</a> <small>(2026-07-05)</small></li>
    <li><a href="post.html?post=hello-compost">Hello Compost</a> <small>(July 05, 2026)</small></li>
  </ul>
</p>


## 5. Serve the site locally

This starts a tiny web server (`python -m http.server`) on your own
machine, so you can view the site in a browser exactly like a visitor
would - just at `http://localhost:8080/` instead of a real domain. It
runs in the background so the notebook can keep going; we'll stop it in
step 7.

In [9]:
import time

server = subprocess.Popen(
    ["python3", "-m", "http.server", "8080"],
    cwd=os.path.join(VANILLA_COMPOST, "src"),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)  # give it a moment to start
print(f"Server started (pid {server.pid}) at http://localhost:8080/")

Server started (pid 3016) at http://localhost:8080/


## 6. Verify it's serving the update correctly

This is the actual "green test": fetch the page from the server we just
started and check it matches what `generate_posts.py` wrote, and that the
sample post shows up. If both checks pass, you'll see a green message
below - proof the whole chain (write markdown -> generate HTML -> serve
it) works end to end on your machine.

In [10]:
import urllib.request

with urllib.request.urlopen("http://localhost:8080/posts.html") as response:
    served = response.read().decode("utf-8")

assert served == generated, "Served posts.html doesn't match what generate_posts.py just wrote."
assert "hello-compost" in served, "Expected the hello-compost post to be listed."

print("Green: the server is serving the freshly generated posts.html.")

Green: the server is serving the freshly generated posts.html.


## 7. Clean up

In [ ]:
server.terminate()
server.wait()
print("Server stopped.")

If everything above ran without errors, python is bootstrapped and working
end to end on this machine, verified against a small app instead of just
assumed to be working. See `greenTest-Message.md` for a running history of these runs.

From here, `$VANILLA_COMPOST/README.md` picks up: edit its `src/posts/`
with your own content, push it to your own GitHub repo, and host it for
free on Netlify. See `../ECOLOGY.md` for how this fits into the rest of
the Ecology Computing methodology.